In [ ]:
# Cell 1 - Install dependencies
# Run this cell once per kernel if dependencies are missing.
%pip install -q python-dotenv pandas requests tenacity snowflake-connector-python[pandas] ipywidgets


In [ ]:
# Cell 2 - Helpers and configuration
import getpass
import html
import json
import logging
import os
from datetime import date, datetime, timezone
from decimal import Decimal
from email.utils import parsedate_to_datetime
from pathlib import Path
from typing import Optional
from urllib.parse import parse_qsl, parse_qs, urlencode, urlparse, urlunparse

import pandas as pd
import requests
from dotenv import load_dotenv
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)


NOTEBOOK_FILENAME = "snowflake_trust_center_to_secops.ipynb"
ACCOUNT_ENV_PREFIXES = (
    "SNOWFLAKE_ACCOUNT_",
    "SNOWFLAKE_USER_",
    "SNOWFLAKE_PRIVATE_KEY_PATH_",
    "SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_",
    "SNOWFLAKE_ROLE_",
    "SNOWFLAKE_WAREHOUSE_",
    "SNOWFLAKE_LABEL_",
)
SECURITY_CHECK_ENV_PREFIXES = (
    "SECURITY_CHECK_NAME_",
    "SECURITY_CHECK_SQL_",
)
TIMESTAMP_FIELD_CANDIDATES = (
    "event_timestamp",
    "timestamp",
    "occurred_at",
    "completed_at",
    "created_at",
    "updated_at",
)
_MAX_REQUEST_BYTES = (4 * 1024 * 1024) - 1024
logging.basicConfig(level=logging.WARNING)
_log = logging.getLogger("secops_sender")
_DEFAULT_SECOPS_WAIT = wait_exponential(multiplier=1, min=2, max=30)


class RetryableSecOpsError(Exception):
    """Raised for retryable webhook responses."""

    def __init__(self, message: str, retry_after: Optional[int] = None):
        super().__init__(message)
        self.retry_after = retry_after


class NonRetryableSecOpsError(Exception):
    """Raised for permanent webhook failures."""


def detect_project_dir() -> Path:
    """Resolve the notebook directory or fall back to the current working directory."""
    override = os.getenv("NOTEBOOK_ROOT", "").strip()
    if override:
        return Path(override).expanduser().resolve()

    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        if (root / NOTEBOOK_FILENAME).exists():
            return root.resolve()
    return Path.cwd().resolve()


def resolve_env_path(project_dir: Optional[Path] = None) -> Path:
    """Resolve the configuration file path."""
    project_dir = project_dir or detect_project_dir()
    env_path = os.getenv("CONFIG_ENV_PATH", str(project_dir / ".env"))
    return Path(env_path).expanduser().resolve()


def load_environment(env_path: Optional[Path] = None) -> tuple[Path, Path]:
    """Load `.env` into the process environment and return the resolved paths."""
    project_dir = detect_project_dir()
    resolved_env_path = Path(env_path or resolve_env_path(project_dir)).expanduser().resolve()
    load_dotenv(resolved_env_path, override=True)
    return project_dir, resolved_env_path


def normalize_private_key_path(raw_path: str) -> str:
    """Normalize the configured private key path for display and connector use."""
    raw_path = (raw_path or "").strip()
    if not raw_path:
        return ""
    return str(Path(raw_path).expanduser().resolve())


def decode_multiline_env_value(raw_value: str) -> str:
    """Turn escaped line breaks from `.env` into real newlines."""
    value = (raw_value or "").strip()
    if not value:
        return ""
    return value.replace("\\n", "\n").replace("\\t", "\t")


def collect_suffix_indexes(environ: dict, prefixes: tuple[str, ...]) -> list[int]:
    """Collect numeric suffixes for the provided environment variable prefixes."""
    indexes = set()
    for key in environ:
        for prefix in prefixes:
            if key.startswith(prefix):
                suffix = key[len(prefix) :]
                if suffix.isdigit():
                    indexes.add(int(suffix))
    return sorted(indexes)


def build_accounts(environ: dict, prompt_for_missing: bool = True) -> list[dict]:
    """Build Snowflake account configuration from environment variables."""
    indexes = collect_suffix_indexes(environ, ACCOUNT_ENV_PREFIXES)
    accounts = []
    for idx in indexes:
        account = environ.get(f"SNOWFLAKE_ACCOUNT_{idx}", "").strip()
        user = environ.get(f"SNOWFLAKE_USER_{idx}", "").strip()
        private_key_path = normalize_private_key_path(
            environ.get(f"SNOWFLAKE_PRIVATE_KEY_PATH_{idx}", "")
        )
        private_key_passphrase = environ.get(f"SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_{idx}", "")
        role = environ.get(f"SNOWFLAKE_ROLE_{idx}", "").strip()
        warehouse = environ.get(f"SNOWFLAKE_WAREHOUSE_{idx}", "").strip()
        label = environ.get(f"SNOWFLAKE_LABEL_{idx}", "").strip() or f"account_{idx}"

        if not account:
            continue

        missing = []
        if not user:
            missing.append(f"SNOWFLAKE_USER_{idx}")
        if not private_key_path:
            missing.append(f"SNOWFLAKE_PRIVATE_KEY_PATH_{idx}")
        if missing:
            raise ValueError(
                f"Snowflake account {idx} is missing required values: {', '.join(missing)}"
            )

        accounts.append(
            {
                "index": idx,
                "key": f"account_{idx}",
                "label": label,
                "account": account,
                "user": user,
                "private_key_path": private_key_path,
                "private_key_passphrase": private_key_passphrase,
                "role": role,
                "warehouse": warehouse,
            }
        )

    if accounts:
        return accounts
    if not prompt_for_missing:
        raise ValueError("At least one Snowflake account is required.")

    account = input("Snowflake account locator: ").strip()
    user = input("Snowflake user: ").strip()
    private_key_path = normalize_private_key_path(input("Snowflake private key path: ").strip())
    private_key_passphrase = getpass.getpass(
        "Snowflake private key passphrase (leave blank if not encrypted): "
    )
    if not account or not user or not private_key_path:
        raise ValueError("Snowflake account, user, and private key path are required.")

    return [
        {
            "index": 1,
            "key": "account_1",
            "label": "account_1",
            "account": account,
            "user": user,
            "private_key_path": private_key_path,
            "private_key_passphrase": private_key_passphrase,
            "role": "",
            "warehouse": "",
        }
    ]


def build_security_checks(environ: dict) -> list[dict]:
    """Build configured SQL checks from environment variables."""
    indexes = collect_suffix_indexes(environ, SECURITY_CHECK_ENV_PREFIXES)
    checks = []
    for idx in indexes:
        raw_sql = environ.get(f"SECURITY_CHECK_SQL_{idx}", "")
        query_name = environ.get(f"SECURITY_CHECK_NAME_{idx}", "").strip() or f"security_check_{idx}"
        sql = decode_multiline_env_value(raw_sql)

        if not sql:
            if environ.get(f"SECURITY_CHECK_NAME_{idx}", "").strip():
                raise ValueError(
                    f"SECURITY_CHECK_NAME_{idx} is set but SECURITY_CHECK_SQL_{idx} is blank."
                )
            continue

        checks.append(
            {
                "index": idx,
                "key": f"security_check_{idx}",
                "name": query_name,
                "sql": sql,
            }
        )

    if not checks:
        raise ValueError("At least one SECURITY_CHECK_SQL_<n> value is required.")
    return checks


def url_has_query_credential(url: str, key: str) -> bool:
    """Return True when the webhook URL already carries the requested auth parameter."""
    values = parse_qs(urlparse(url).query).get(key, [])
    return any(value.strip() for value in values)


def describe_auth_mode(url: str) -> str:
    """Describe how the webhook is authenticated."""
    has_key = url_has_query_credential(url, "key")
    has_secret = url_has_query_credential(url, "secret")
    if has_key and has_secret:
        return "url_query"
    if (not has_key) and (not has_secret):
        return "headers"
    return "mixed"


def mask_webhook_url(url: str) -> str:
    """Mask key and secret values when rendering the webhook URL."""
    if not url:
        return "<unset>"
    parsed = urlparse(url)
    masked_items = []
    for item_key, item_value in parse_qsl(parsed.query, keep_blank_values=True):
        if item_key in {"key", "secret"} and item_value:
            masked_items.append((item_key, "***"))
        else:
            masked_items.append((item_key, item_value))
    return urlunparse(parsed._replace(query=urlencode(masked_items)))


def build_runtime_config(
    environ: dict,
    project_dir: Path,
    env_path: Path,
    prompt_for_missing: bool = True,
) -> dict:
    """Build the notebook runtime configuration."""
    accounts = build_accounts(environ, prompt_for_missing=prompt_for_missing)
    security_checks = build_security_checks(environ)

    webhook_url = environ.get("SECOPS_WEBHOOK_URL", "").strip()
    if not webhook_url and prompt_for_missing:
        webhook_url = input("Google SecOps webhook URL: ").strip()
    if not webhook_url:
        raise ValueError("SECOPS_WEBHOOK_URL is required.")

    config = {
        "PROJECT_DIR": str(project_dir),
        "ENV_PATH": str(env_path),
        "ENV_FOUND": env_path.exists(),
        "SNOWFLAKE_ACCOUNTS": accounts,
        "SECURITY_CHECKS": security_checks,
        "SECOPS_WEBHOOK_URL": webhook_url,
        "SECOPS_API_KEY": environ.get("SECOPS_API_KEY", "").strip(),
        "SECOPS_WEBHOOK_SECRET": environ.get("SECOPS_WEBHOOK_SECRET", "").strip(),
        "BATCH_SIZE": int(environ.get("BATCH_SIZE", "100")),
        "DRY_RUN": environ.get("DRY_RUN", "false").strip().lower() == "true",
        "QUERY_RESULTS": {},
        "SEND_SELECTION": [],
    }

    if not config["DRY_RUN"] and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key"):
        if prompt_for_missing and not config["SECOPS_API_KEY"]:
            config["SECOPS_API_KEY"] = getpass.getpass("Google SecOps API key: ")
    if not config["DRY_RUN"] and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret"):
        if prompt_for_missing and not config["SECOPS_WEBHOOK_SECRET"]:
            config["SECOPS_WEBHOOK_SECRET"] = getpass.getpass("Google SecOps webhook secret: ")

    missing_auth = []
    if (
        not config["DRY_RUN"]
        and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key")
        and not config["SECOPS_API_KEY"]
    ):
        missing_auth.append("SECOPS_API_KEY")
    if (
        not config["DRY_RUN"]
        and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret")
        and not config["SECOPS_WEBHOOK_SECRET"]
    ):
        missing_auth.append("SECOPS_WEBHOOK_SECRET")
    if missing_auth:
        raise ValueError("Missing Google SecOps auth values: " + ", ".join(missing_auth))

    return config


def summarize_runtime_config(config: dict) -> dict:
    """Build a concise configuration summary for notebook output."""
    return {
        "project_dir": config["PROJECT_DIR"],
        "env_path": config["ENV_PATH"],
        "env_found": config["ENV_FOUND"],
        "accounts": [
            {
                "label": account["label"],
                "account": account["account"],
                "user": account["user"],
                "role": account["role"] or "<default>",
                "warehouse": account["warehouse"] or "<default>",
            }
            for account in config["SNOWFLAKE_ACCOUNTS"]
        ],
        "security_checks": [
            {
                "key": check["key"],
                "name": check["name"],
                "sql_lines": len(check["sql"].splitlines()) or 1,
            }
            for check in config["SECURITY_CHECKS"]
        ],
        "batch_size": config["BATCH_SIZE"],
        "dry_run_default": config["DRY_RUN"],
        "secops_auth_mode": describe_auth_mode(config["SECOPS_WEBHOOK_URL"]),
        "secops_webhook_url": mask_webhook_url(config["SECOPS_WEBHOOK_URL"]),
    }


def make_group_key(account_label: str, query_key: str) -> str:
    """Build a stable key for one account/query group."""
    return f"{account_label}::{query_key}"


def split_group_key(group_key: str) -> tuple[str, str]:
    """Split one account/query group key."""
    account_label, query_key = group_key.split("::", 1)
    return account_label, query_key


def build_group_catalog(config: dict) -> dict:
    """Build the account/query matrix metadata."""
    group_lookup = {}
    group_order = []
    for account in config["SNOWFLAKE_ACCOUNTS"]:
        for query in config["SECURITY_CHECKS"]:
            group_key = make_group_key(account["label"], query["key"])
            group_meta = {
                "group_key": group_key,
                "account_label": account["label"],
                "query_key": query["key"],
                "query_name": query["name"],
                "account": account,
                "query": query,
            }
            group_lookup[group_key] = group_meta
            group_order.append(group_key)

    return {
        "accounts": config["SNOWFLAKE_ACCOUNTS"],
        "queries": config["SECURITY_CHECKS"],
        "group_lookup": group_lookup,
        "group_order": group_order,
    }


def normalize_group_keys(group_catalog: dict, group_keys: list[str]) -> list[str]:
    """Normalize group selection order and validate keys."""
    requested = list(group_keys or [])
    invalid = sorted(set(requested) - set(group_catalog["group_lookup"]))
    if invalid:
        raise ValueError(f"Unknown group keys: {invalid}")

    selected = []
    requested_set = set(requested)
    for group_key in group_catalog["group_order"]:
        if group_key in requested_set:
            selected.append(group_key)
    return selected


def build_default_checked_groups(config: dict, group_catalog: Optional[dict] = None) -> list[str]:
    """Select the first account and all configured queries by default."""
    group_catalog = group_catalog or build_group_catalog(config)
    if not group_catalog["accounts"]:
        return []

    first_account = group_catalog["accounts"][0]["label"]
    selected = []
    for query in group_catalog["queries"]:
        selected.append(make_group_key(first_account, query["key"]))
    return selected


def build_default_selection(config: dict) -> dict:
    """Build the default account/query selection summary."""
    default_accounts = []
    if config["SNOWFLAKE_ACCOUNTS"]:
        default_accounts = [config["SNOWFLAKE_ACCOUNTS"][0]["label"]]
    return {
        "accounts": default_accounts,
        "queries": [check["key"] for check in config["SECURITY_CHECKS"]],
    }


def build_empty_query_result(group_meta: dict) -> dict:
    """Create the default result object for one matrix cell."""
    return {
        "group_key": group_meta["group_key"],
        "account_label": group_meta["account_label"],
        "query_key": group_meta["query_key"],
        "query_name": group_meta["query_name"],
        "status": "pending",
        "row_count": 0,
        "columns": [],
        "dataframe": pd.DataFrame(),
        "error": None,
    }


def build_initial_query_results(config: dict, group_catalog: Optional[dict] = None) -> dict:
    """Build an empty result object for every account/query cell."""
    group_catalog = group_catalog or build_group_catalog(config)
    return {
        group_key: build_empty_query_result(group_meta)
        for group_key, group_meta in group_catalog["group_lookup"].items()
    }


def get_selection_snapshot(
    checked_groups: list[str],
    dry_run: bool,
    group_catalog: dict,
) -> dict:
    """Build a normalized runtime selection snapshot."""
    return {
        "checked_groups": normalize_group_keys(group_catalog, checked_groups),
        "dry_run": bool(dry_run),
    }


def sync_config_runtime_fields(config: dict, app_state: dict) -> None:
    """Mirror runtime state onto CONFIG fields used elsewhere in the notebook."""
    config["QUERY_RESULTS"] = app_state["query_results"]
    config["SEND_SELECTION"] = list(app_state["checked_groups"])


def create_app_state(config: dict) -> dict:
    """Create notebook runtime state."""
    group_catalog = build_group_catalog(config)
    app_state = {
        "group_catalog": group_catalog,
        "checked_groups": build_default_checked_groups(config, group_catalog),
        "dry_run": config["DRY_RUN"],
        "last_run_selection": None,
        "selection_dirty": False,
        "dirty_reason": "Run selected groups first.",
        "sanity_check_results": [],
        "query_results": build_initial_query_results(config, group_catalog),
        "payload_plan": {
            "selected_groups": [],
            "total_events": 0,
            "generated_at": None,
            "dry_run": config["DRY_RUN"],
        },
        "selected_events": [],
        "secops_result": None,
    }
    sync_config_runtime_fields(config, app_state)
    return app_state


def sync_runtime_selection(app_state: dict, checked_groups: list[str], dry_run: bool) -> dict:
    """Update runtime selection and compute whether a rerun is required."""
    snapshot = get_selection_snapshot(checked_groups, dry_run, app_state["group_catalog"])
    previous_run = app_state.get("last_run_selection")

    app_state["checked_groups"] = snapshot["checked_groups"]
    app_state["dry_run"] = snapshot["dry_run"]

    if previous_run is None:
        app_state["selection_dirty"] = False
        app_state["dirty_reason"] = "Run selected groups first."
        return snapshot

    selection_changed = previous_run["checked_groups"] != snapshot["checked_groups"]
    mode_changed = previous_run["dry_run"] != snapshot["dry_run"]
    if selection_changed or mode_changed:
        app_state["selection_dirty"] = True
        if selection_changed and mode_changed:
            app_state["dirty_reason"] = "Selection changed and mode changed; rerun required before send."
        elif selection_changed:
            app_state["dirty_reason"] = "Selection changed; rerun required before send."
        else:
            app_state["dirty_reason"] = "Mode changed; rerun required before send."
    else:
        app_state["selection_dirty"] = False
        app_state["dirty_reason"] = "Ready to send."

    return snapshot


def get_selected_accounts(checked_groups: list[str]) -> list[str]:
    """Return selected account labels in matrix order."""
    selected = []
    seen = set()
    for group_key in checked_groups:
        account_label, _query_key = split_group_key(group_key)
        if account_label not in seen:
            seen.add(account_label)
            selected.append(account_label)
    return selected


def reduce_send_state(app_state: dict) -> dict:
    """Determine whether the current runtime selection can be sent to SecOps."""
    eligible_groups = []
    for group_key in app_state["checked_groups"]:
        result = app_state["query_results"].get(group_key)
        if not result:
            continue
        if result["status"] == "success" and result["row_count"] > 0:
            eligible_groups.append(group_key)

    if not app_state["checked_groups"]:
        return {
            "enabled": False,
            "reason": "Select at least one account/query group.",
            "eligible_groups": eligible_groups,
        }
    if app_state["last_run_selection"] is None:
        return {
            "enabled": False,
            "reason": "Run selected groups first.",
            "eligible_groups": eligible_groups,
        }
    if app_state["selection_dirty"]:
        return {
            "enabled": False,
            "reason": app_state["dirty_reason"],
            "eligible_groups": eligible_groups,
        }
    if not eligible_groups:
        return {
            "enabled": False,
            "reason": "No checked groups have successful non-empty results.",
            "eligible_groups": eligible_groups,
        }
    return {
        "enabled": True,
        "reason": f"{len(eligible_groups)} checked group(s) ready to send.",
        "eligible_groups": eligible_groups,
    }


def summarize_query_results(query_results: dict) -> dict:
    """Summarize query execution outcomes."""
    summary = {
        "groups": len(query_results),
        "pending": 0,
        "running": 0,
        "success": 0,
        "empty": 0,
        "error": 0,
        "rows": 0,
    }
    for result in query_results.values():
        status = result["status"]
        if status in summary:
            summary[status] += 1
        summary["rows"] += result.get("row_count", 0)
    return summary


def build_header_snapshot(config: dict, app_state: dict) -> dict:
    """Build the compact control-panel header state."""
    send_state = reduce_send_state(app_state)
    return {
        "selected_group_count": len(app_state["checked_groups"]),
        "eligible_group_count": len(send_state["eligible_groups"]),
        "dry_run": app_state["dry_run"],
        "auth_mode": describe_auth_mode(config["SECOPS_WEBHOOK_URL"]),
        "masked_webhook_url": mask_webhook_url(config["SECOPS_WEBHOOK_URL"]),
        "selection_dirty": app_state["selection_dirty"],
        "dirty_reason": app_state["dirty_reason"],
        "send_enabled": send_state["enabled"],
        "send_reason": send_state["reason"],
    }


def build_status_badge(result: dict) -> str:
    """Render one small HTML status badge for the matrix."""
    status = result.get("status", "pending")
    if status == "success":
        label = f"ok:{result.get('row_count', 0)}"
        color = "#0f766e"
        accent = "#ccfbf1"
        detail = f"{result.get('row_count', 0)} row(s)"
    elif status == "empty":
        label = "empty"
        color = "#92400e"
        accent = "#fef3c7"
        detail = "0 row(s)"
    elif status == "error":
        label = "error"
        color = "#b91c1c"
        accent = "#fee2e2"
        detail = html.escape((result.get("error") or "")[:120])
    elif status == "running":
        label = "running"
        color = "#1d4ed8"
        accent = "#dbeafe"
        detail = "Executing"
    else:
        label = "pending"
        color = "#475569"
        accent = "#e2e8f0"
        detail = "Waiting"

    return (
        f"<div style='display:inline-block;padding:2px 8px;border-radius:999px;"
        f"background:{accent};color:{color};font-size:11px;font-weight:600'>{label}</div>"
        f"<div style='margin-top:4px;font-size:11px;color:#475569'>{detail}</div>"
    )


def validate_account_labels(config: dict, account_labels: list[str]) -> None:
    """Validate selected account labels."""
    available_accounts = {account["label"] for account in config["SNOWFLAKE_ACCOUNTS"]}
    invalid_accounts = sorted(set(account_labels) - available_accounts)
    if invalid_accounts:
        raise ValueError(f"Unknown Snowflake account labels: {invalid_accounts}")


def connect_snowflake(acct_cfg: dict):
    """Create a Snowflake connection using key pair authentication."""
    from pathlib import Path as _Path

    import snowflake.connector

    private_key_path = _Path(acct_cfg["private_key_path"]).expanduser().resolve()
    if not private_key_path.exists():
        raise FileNotFoundError(
            f"[{acct_cfg['label']}] Private key file not found: {private_key_path}"
        )

    connect_kwargs = {
        "account": acct_cfg["account"],
        "user": acct_cfg["user"],
        "authenticator": "SNOWFLAKE_JWT",
        "private_key_file": str(private_key_path),
        "private_key_file_pwd": acct_cfg.get("private_key_passphrase") or None,
        "client_session_keep_alive": False,
        "network_timeout": 30,
    }
    if acct_cfg.get("role"):
        connect_kwargs["role"] = acct_cfg["role"]
    if acct_cfg.get("warehouse"):
        connect_kwargs["warehouse"] = acct_cfg["warehouse"]
    return snowflake.connector.connect(**connect_kwargs)


def close_connection(conn) -> None:
    """Close a Snowflake connection when possible."""
    if conn is None:
        return
    close = getattr(conn, "close", None)
    if callable(close):
        close()


def fetch_query_dataframe(conn, sql: str) -> pd.DataFrame:
    """Execute one SQL statement and return the result as a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql)
        if not cur.description:
            return pd.DataFrame()
        columns = [column[0] for column in cur.description]
        frames = []
        try:
            for batch in cur.fetch_pandas_batches():
                frames.append(batch.reindex(columns=columns))
        except Exception:
            rows = cur.fetchall()
            frames = [pd.DataFrame(rows, columns=columns)]
    if not frames:
        return pd.DataFrame(columns=columns)
    return pd.concat(frames, ignore_index=True).reindex(columns=columns)


def run_selected_account_sanity_checks(
    config: dict,
    account_labels: list[str],
    connect_fn=None,
    close_fn=None,
) -> list[dict]:
    """Run a basic session query for the selected accounts."""
    validate_account_labels(config, account_labels)
    account_lookup = {account["label"]: account for account in config["SNOWFLAKE_ACCOUNTS"]}
    connect_fn = connect_fn or connect_snowflake
    close_fn = close_fn or close_connection
    results = []
    for account_label in account_labels:
        conn = None
        try:
            conn = connect_fn(account_lookup[account_label])
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT CURRENT_ACCOUNT(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_VERSION()"
                )
                current_account, current_role, current_warehouse, current_version = cur.fetchone()
            results.append(
                {
                    "account_label": account_label,
                    "status": "success",
                    "current_account": current_account,
                    "current_role": current_role,
                    "current_warehouse": current_warehouse,
                    "current_version": current_version,
                }
            )
        except Exception as exc:
            results.append(
                {
                    "account_label": account_label,
                    "status": "error",
                    "error": str(exc),
                }
            )
        finally:
            close_fn(conn)
    return results


def execute_selected_groups(
    config: dict,
    selected_group_keys: list[str],
    connect_fn=None,
    fetch_fn=None,
    close_fn=None,
    progress_fn=None,
) -> dict:
    """Execute the selected account/query cells."""
    group_catalog = build_group_catalog(config)
    selected_group_keys = normalize_group_keys(group_catalog, selected_group_keys)
    connect_fn = connect_fn or connect_snowflake
    fetch_fn = fetch_fn or fetch_query_dataframe
    close_fn = close_fn or close_connection

    results = {}
    for group_key in selected_group_keys:
        group_meta = group_catalog["group_lookup"][group_key]
        result = build_empty_query_result(group_meta)
        result["status"] = "running"
        if progress_fn:
            progress_fn(group_key, result.copy())

        conn = None
        try:
            conn = connect_fn(group_meta["account"])
            dataframe = fetch_fn(conn, group_meta["query"]["sql"]).copy()
            result["dataframe"] = dataframe
            result["columns"] = dataframe.columns.tolist()
            result["row_count"] = len(dataframe)
            result["status"] = "empty" if dataframe.empty else "success"
        except Exception as exc:
            result["status"] = "error"
            result["error"] = str(exc)
        finally:
            close_fn(conn)

        results[group_key] = result
        if progress_fn:
            progress_fn(group_key, result.copy())
    return results


def json_safe_value(value):
    """Convert notebook values into JSON-safe primitives."""
    if value is None or value is pd.NaT:
        return None
    try:
        if pd.isna(value) and not isinstance(value, (dict, list, tuple, set)):
            return None
    except Exception:
        pass
    if isinstance(value, pd.Timestamp):
        if value.tzinfo is None:
            value = value.tz_localize(timezone.utc)
        return value.tz_convert(timezone.utc).isoformat()
    if isinstance(value, datetime):
        if value.tzinfo is None:
            value = value.replace(tzinfo=timezone.utc)
        return value.astimezone(timezone.utc).isoformat()
    if isinstance(value, date):
        return value.isoformat()
    if isinstance(value, Decimal):
        if value == value.to_integral():
            return int(value)
        return float(value)
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, dict):
        return {str(key): json_safe_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe_value(item) for item in value]
    item = getattr(value, "item", None)
    if callable(item):
        try:
            unpacked = item()
        except Exception:
            unpacked = value
        if unpacked is not value:
            return json_safe_value(unpacked)
    return value


def extract_source_timestamp(record: dict) -> Optional[str]:
    """Extract a source timestamp from a result record when one is available."""
    lower_map = {str(key).lower(): key for key in record}
    for candidate in TIMESTAMP_FIELD_CANDIDATES:
        original_key = lower_map.get(candidate)
        if original_key is None:
            continue
        parsed = pd.to_datetime(record.get(original_key), utc=True, errors="coerce")
        if pd.isna(parsed):
            continue
        return parsed.isoformat()
    return None


def row_to_generic_log(
    record: dict,
    account_label: str,
    query_key: str,
    query_name: str,
    row_index: int,
    generated_at: Optional[str] = None,
) -> dict:
    """Convert one query result row into a generic SecOps log object."""
    generated_at = generated_at or datetime.now(timezone.utc).isoformat()
    safe_record = {str(key): json_safe_value(value) for key, value in record.items()}
    event = {
        "snowflake_account": account_label,
        "snowflake_query_key": query_key,
        "snowflake_query_name": query_name,
        "generated_at": generated_at,
        "row_index": row_index,
        "result": safe_record,
    }
    source_timestamp = extract_source_timestamp(record)
    if source_timestamp:
        event["source_timestamp"] = source_timestamp
    return event


def build_group_events(query_result: dict, generated_at: Optional[str] = None) -> list[dict]:
    """Convert one result group into generic SecOps log objects."""
    dataframe = query_result["dataframe"]
    if dataframe.empty:
        return []
    generated_at = generated_at or datetime.now(timezone.utc).isoformat()
    events = []
    for row_index, record in enumerate(dataframe.to_dict(orient="records"), start=1):
        events.append(
            row_to_generic_log(
                record,
                account_label=query_result["account_label"],
                query_key=query_result["query_key"],
                query_name=query_result["query_name"],
                row_index=row_index,
                generated_at=generated_at,
            )
        )
    return events


def build_selected_events(
    query_results: dict,
    selected_group_keys: list[str],
    dry_run: Optional[bool] = None,
) -> tuple[list[dict], dict]:
    """Build payload events for the checked groups."""
    generated_at = datetime.now(timezone.utc).isoformat()
    events = []
    group_summaries = []
    for group_key in selected_group_keys:
        result = query_results.get(group_key)
        if not result:
            continue
        if result["status"] != "success" or result["row_count"] == 0:
            continue
        group_events = build_group_events(result, generated_at=generated_at)
        events.extend(group_events)
        group_summaries.append(
            {
                "group_key": group_key,
                "account_label": result["account_label"],
                "query_key": result["query_key"],
                "query_name": result["query_name"],
                "row_count": len(group_events),
            }
        )
    return events, {
        "selected_groups": group_summaries,
        "total_events": len(events),
        "generated_at": generated_at,
        "dry_run": dry_run,
    }


def build_sender_config(config: dict, app_state: dict) -> dict:
    """Build the runtime sender config using the UI dry-run toggle."""
    sender_config = dict(config)
    sender_config["DRY_RUN"] = bool(app_state["dry_run"])
    return sender_config


def _parse_retry_after(value: Optional[str]) -> Optional[int]:
    """Parse the HTTP Retry-After header."""
    if not value:
        return None
    try:
        return max(0, int(value))
    except ValueError:
        pass
    try:
        retry_at = parsedate_to_datetime(value)
        if retry_at.tzinfo is None:
            retry_at = retry_at.replace(tzinfo=timezone.utc)
        return max(0, int((retry_at - datetime.now(timezone.utc)).total_seconds()))
    except Exception:
        return None


def _secops_wait(retry_state) -> float:
    """Use Retry-After when it is available."""
    exc = retry_state.outcome.exception() if retry_state.outcome else None
    if isinstance(exc, RetryableSecOpsError) and exc.retry_after is not None:
        return exc.retry_after
    return _DEFAULT_SECOPS_WAIT(retry_state)


def _build_headers(config: dict) -> dict:
    """Build SecOps webhook headers."""
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "snowflake-security-checks-secops/1.0",
    }
    if not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key"):
        if not config.get("SECOPS_API_KEY"):
            raise NonRetryableSecOpsError(
                "SECOPS_API_KEY is required because the webhook URL does not include key=."
            )
        headers["X-goog-api-key"] = config["SECOPS_API_KEY"]
    if not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret"):
        if not config.get("SECOPS_WEBHOOK_SECRET"):
            raise NonRetryableSecOpsError(
                "SECOPS_WEBHOOK_SECRET is required because the webhook URL does not include secret=."
            )
        headers["X-Webhook-Access-Key"] = config["SECOPS_WEBHOOK_SECRET"]
    return headers


def _serialize_event(event: dict) -> bytes:
    """Serialize one event as JSON bytes."""
    return json.dumps(event, separators=(",", ":"), ensure_ascii=False).encode("utf-8")


def _build_event_batches(events: list[dict], batch_size: int) -> list[dict]:
    """Split events by count and payload size."""
    if batch_size < 1:
        raise ValueError("BATCH_SIZE must be >= 1.")

    batches = []
    current_lines = []
    current_bytes = 0
    for event_index, event in enumerate(events, start=1):
        line = _serialize_event(event)
        if len(line) > _MAX_REQUEST_BYTES:
            raise NonRetryableSecOpsError(
                f"Event {event_index} is {len(line)} bytes, exceeding the 4 MB webhook limit."
            )
        line_bytes = len(line) + (1 if current_lines else 0)
        if current_lines and (
            len(current_lines) >= batch_size or current_bytes + line_bytes > _MAX_REQUEST_BYTES
        ):
            payload = b"\n".join(current_lines)
            batches.append(
                {
                    "events": len(current_lines),
                    "bytes": len(payload),
                    "payload": payload,
                }
            )
            current_lines = [line]
            current_bytes = len(line)
        else:
            current_lines.append(line)
            current_bytes += line_bytes

    if current_lines:
        payload = b"\n".join(current_lines)
        batches.append(
            {
                "events": len(current_lines),
                "bytes": len(payload),
                "payload": payload,
            }
        )
    return batches


@retry(
    stop=stop_after_attempt(4),
    wait=_secops_wait,
    retry=retry_if_exception_type((requests.RequestException, RetryableSecOpsError)),
    before_sleep=before_sleep_log(_log, logging.WARNING),
    reraise=True,
)
def _post_batch(url: str, headers: dict, payload: bytes, post_fn=requests.post):
    """POST one NDJSON batch to Google SecOps."""
    response = post_fn(url, headers=headers, data=payload, timeout=60)
    if response.status_code in (408, 429) or response.status_code >= 500:
        retry_after = _parse_retry_after(response.headers.get("Retry-After"))
        raise RetryableSecOpsError(
            f"Retryable error {response.status_code}: {response.text[:300]}",
            retry_after=retry_after,
        )
    if response.status_code >= 400:
        raise NonRetryableSecOpsError(
            f"Client error {response.status_code}: {response.text[:500]}\\n"
            "  -> 401/403: check webhook URL query auth or fallback headers\\n"
            "  -> 413: reduce BATCH_SIZE or trim oversized events"
        )
    return response


def send_to_secops(events: list[dict], config: dict, post_fn=requests.post) -> dict:
    """Send the prepared events to Google SecOps or render a dry-run preview."""
    total = len(events)
    auth_mode = describe_auth_mode(config["SECOPS_WEBHOOK_URL"])
    if total == 0:
        print("No events selected for Google SecOps delivery.")
        return {
            "sent": 0,
            "batches": 0,
            "attempted_batches": 0,
            "failed_batches": 0,
            "failed_event_count": 0,
            "planned_batches": 0,
            "planned_bytes": 0,
            "auth_mode": auth_mode,
            "dry_run": config["DRY_RUN"],
            "skipped": True,
            "errors": [],
        }

    batches = _build_event_batches(events, config["BATCH_SIZE"])
    planned_bytes = sum(batch["bytes"] for batch in batches)

    if config["DRY_RUN"]:
        first_batch = batches[0]
        preview = first_batch["payload"][:1500].decode("utf-8", errors="replace")
        print(f"[DRY_RUN] {total} event(s) would be sent to {mask_webhook_url(config['SECOPS_WEBHOOK_URL'])}")
        print(f"Auth mode   : {auth_mode}")
        print(f"Batches     : {len(batches)}")
        print(f"Total bytes : {planned_bytes}")
        print(f"First batch : {first_batch['events']} event(s) / {first_batch['bytes']} bytes")
        print("\\nFirst batch preview:")
        print(preview)
        if len(first_batch["payload"]) > 1500:
            print("... <truncated>")
        return {
            "sent": 0,
            "batches": 0,
            "attempted_batches": 0,
            "failed_batches": 0,
            "failed_event_count": 0,
            "planned_batches": len(batches),
            "planned_bytes": planned_bytes,
            "auth_mode": auth_mode,
            "dry_run": True,
            "skipped": False,
            "errors": [],
        }

    headers = _build_headers(config)
    sent = 0
    successful_batches = 0
    failed_batches = 0
    failed_event_count = 0
    errors = []
    attempted_batches = 0

    for batch_index, batch in enumerate(batches, start=1):
        attempted_batches = batch_index
        try:
            _post_batch(config["SECOPS_WEBHOOK_URL"], headers, batch["payload"], post_fn=post_fn)
            sent += batch["events"]
            successful_batches += 1
            print(
                f"  Batch {batch_index}: sent {batch['events']} event(s) / {batch['bytes']} bytes "
                f"(total {sent}/{total})"
            )
        except Exception as exc:
            failed_batches += 1
            failed_event_count += batch["events"]
            errors.append(
                {
                    "batch": batch_index,
                    "events": batch["events"],
                    "bytes": batch["bytes"],
                    "error": str(exc),
                }
            )
            print(f"  Batch {batch_index}: failed after retries - {exc}")
            break

    if failed_batches:
        print(f"Stopped after {sent} sent event(s); {failed_event_count} event(s) remain unsent.")
    else:
        print(f"Complete: {sent} event(s) sent in {successful_batches} batch(es).")

    return {
        "sent": sent,
        "batches": successful_batches,
        "attempted_batches": attempted_batches,
        "failed_batches": failed_batches,
        "failed_event_count": failed_event_count,
        "planned_batches": len(batches),
        "planned_bytes": planned_bytes,
        "auth_mode": auth_mode,
        "dry_run": False,
        "skipped": False,
        "errors": errors,
    }


def build_debug_snapshot(config: dict, app_state: dict) -> dict:
    """Build a JSON-safe debug snapshot for the final notebook cell."""
    query_result_snapshot = {}
    for group_key, result in app_state["query_results"].items():
        query_result_snapshot[group_key] = {
            "account_label": result["account_label"],
            "query_key": result["query_key"],
            "query_name": result["query_name"],
            "status": result["status"],
            "row_count": result["row_count"],
            "columns": list(result.get("columns", [])),
            "error": result.get("error"),
        }

    return {
        "CONFIG": summarize_runtime_config(config),
        "APP_STATE": {
            "checked_groups": list(app_state["checked_groups"]),
            "dry_run": app_state["dry_run"],
            "last_run_selection": json_safe_value(app_state["last_run_selection"]),
            "selection_dirty": app_state["selection_dirty"],
            "dirty_reason": app_state["dirty_reason"],
            "sanity_check_results": json_safe_value(app_state["sanity_check_results"]),
            "header": build_header_snapshot(config, app_state),
            "selected_events": len(app_state["selected_events"]),
        },
        "QUERY_RESULTS": query_result_snapshot,
        "PAYLOAD_PLAN": json_safe_value(app_state["payload_plan"]),
        "SECOPS_RESULT": json_safe_value(app_state["secops_result"]),
    }


if not globals().get("__NOTEBOOK_TEST__", False):
    PROJECT_DIR, ENV_PATH = load_environment()
    CONFIG = build_runtime_config(
        os.environ,
        project_dir=PROJECT_DIR,
        env_path=ENV_PATH,
        prompt_for_missing=True,
    )
    APP_STATE = create_app_state(CONFIG)
    sync_config_runtime_fields(CONFIG, APP_STATE)
    CONFIG_SUMMARY = summarize_runtime_config(CONFIG)
    print(json.dumps(CONFIG_SUMMARY, indent=2))


In [ ]:
# Cell 3 - Unified control panel
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display


GROUP_CATALOG = APP_STATE["group_catalog"]
GROUP_ORDER = GROUP_CATALOG["group_order"]
GROUP_LOOKUP = GROUP_CATALOG["group_lookup"]
MATRIX_CHECKBOXES = {}
MATRIX_STATUS = {}

HEADER_HTML = widgets.HTML()
STATUS_HTML = widgets.HTML()
DRY_RUN_TOGGLE = widgets.Checkbox(
    value=APP_STATE["dry_run"],
    description="DRY_RUN",
    indent=False,
)
RUN_BUTTON = widgets.Button(
    description="Run Selected",
    button_style="primary",
    icon="play",
)
SEND_BUTTON = widgets.Button(
    description="Send Selected to SecOps",
    icon="paper-plane",
)
RUN_OUTPUT = widgets.Output()
PREVIEW_OUTPUT = widgets.Output()
SEND_OUTPUT = widgets.Output()
OUTPUT_TABS = widgets.Tab(children=[RUN_OUTPUT, PREVIEW_OUTPUT, SEND_OUTPUT])
OUTPUT_TABS.set_title(0, "Run Log")
OUTPUT_TABS.set_title(1, "Preview")
OUTPUT_TABS.set_title(2, "SecOps Log")

matrix_rows = len(GROUP_CATALOG["accounts"]) + 1
matrix_cols = len(GROUP_CATALOG["queries"]) + 1
MATRIX = widgets.GridspecLayout(matrix_rows, matrix_cols, grid_gap="8px")
MATRIX[0, 0] = widgets.HTML(
    "<div style='font-weight:700'>Account / Query</div>",
    layout=widgets.Layout(padding="6px"),
)
for column_index, query in enumerate(GROUP_CATALOG["queries"], start=1):
    MATRIX[0, column_index] = widgets.HTML(
        (
            f"<div style='font-weight:700'>{html.escape(query['name'])}</div>"
            f"<div style='font-size:11px;color:#64748b'>{html.escape(query['key'])}</div>"
        ),
        layout=widgets.Layout(padding="6px"),
    )

for row_index, account in enumerate(GROUP_CATALOG["accounts"], start=1):
    MATRIX[row_index, 0] = widgets.HTML(
        (
            f"<div style='font-weight:700'>{html.escape(account['label'])}</div>"
            f"<div style='font-size:11px;color:#64748b'>{html.escape(account['account'])}</div>"
        ),
        layout=widgets.Layout(padding="6px"),
    )
    for column_index, query in enumerate(GROUP_CATALOG["queries"], start=1):
        group_key = make_group_key(account["label"], query["key"])
        checkbox = widgets.Checkbox(
            value=group_key in set(APP_STATE["checked_groups"]),
            description="",
            indent=False,
            layout=widgets.Layout(width="28px"),
        )
        checkbox.tooltip = f"{account['label']} / {query['name']} [{query['key']}]"
        status_html = widgets.HTML(build_status_badge(APP_STATE["query_results"][group_key]))
        MATRIX_CHECKBOXES[group_key] = checkbox
        MATRIX_STATUS[group_key] = status_html
        MATRIX[row_index, column_index] = widgets.VBox(
            [checkbox, status_html],
            layout=widgets.Layout(
                border="1px solid #cbd5e1",
                border_radius="6px",
                padding="6px",
                min_width="170px",
            ),
        )


def collect_checked_groups() -> list[str]:
    """Read checked matrix cells in display order."""
    return [group_key for group_key in GROUP_ORDER if MATRIX_CHECKBOXES[group_key].value]


def render_matrix_statuses() -> None:
    """Render one status badge per matrix cell."""
    for group_key in GROUP_ORDER:
        MATRIX_STATUS[group_key].value = build_status_badge(APP_STATE["query_results"][group_key])


def render_header() -> None:
    """Render the compact control-panel summary and button state."""
    header = build_header_snapshot(CONFIG, APP_STATE)
    mode_label = "DRY_RUN" if header["dry_run"] else "LIVE"
    dirty_label = "yes" if header["selection_dirty"] else "no"
    HEADER_HTML.value = (
        "<div style='border:1px solid #cbd5e1;border-radius:8px;padding:12px;'>"
        f"<div><b>Webhook</b>: {html.escape(header['masked_webhook_url'])}</div>"
        f"<div style='margin-top:6px'><b>Auth</b>: {html.escape(header['auth_mode'])}"
        f" | <b>Mode</b>: {mode_label}"
        f" | <b>Selected</b>: {header['selected_group_count']}"
        f" | <b>Eligible</b>: {header['eligible_group_count']}"
        f" | <b>Dirty</b>: {dirty_label}</div>"
        "</div>"
    )
    STATUS_HTML.value = (
        "<div style='font-size:12px;color:#475569;margin:4px 0 10px 0'>"
        f"{html.escape(header['send_reason'])}</div>"
    )
    RUN_BUTTON.disabled = header["selected_group_count"] == 0
    SEND_BUTTON.disabled = not header["send_enabled"]


def render_preview_output() -> None:
    """Render query results and payload-plan previews."""
    with PREVIEW_OUTPUT:
        clear_output(wait=True)
        if APP_STATE["last_run_selection"] is None:
            print("Run selected groups to preview results.")
            return

        print(
            f"Latest run mode: {'DRY_RUN' if APP_STATE['last_run_selection']['dry_run'] else 'LIVE'}"
        )
        print(f"Latest run groups: {len(APP_STATE['last_run_selection']['checked_groups'])}")
        if APP_STATE["selection_dirty"]:
            print(APP_STATE["dirty_reason"])

        summary_rows = []
        for group_key in GROUP_ORDER:
            result = APP_STATE["query_results"][group_key]
            if result["status"] == "pending" and group_key not in APP_STATE["last_run_selection"]["checked_groups"]:
                continue
            summary_rows.append(
                {
                    "group_key": group_key,
                    "status": result["status"],
                    "rows": result["row_count"],
                    "error": result["error"],
                }
            )
        if summary_rows:
            display(pd.DataFrame(summary_rows))

        print("\nPayload plan:")
        print(json.dumps(json_safe_value(APP_STATE["payload_plan"]), indent=2))

        preview_groups = [
            group_key
            for group_key in APP_STATE["checked_groups"]
            if APP_STATE["query_results"][group_key]["status"] == "success"
            and APP_STATE["query_results"][group_key]["row_count"] > 0
        ]
        if not preview_groups:
            print("\nNo successful non-empty checked groups to preview.")
            return

        for group_key in preview_groups:
            result = APP_STATE["query_results"][group_key]
            display(
                HTML(
                    (
                        f"<h4>{html.escape(result['account_label'])} / "
                        f"{html.escape(result['query_name'])} "
                        f"[{html.escape(result['query_key'])}]</h4>"
                    )
                )
            )
            display(result["dataframe"].head())


def refresh_selection_state(_change=None) -> None:
    """Sync widget state into APP_STATE and rerender the panel."""
    sync_runtime_selection(APP_STATE, collect_checked_groups(), DRY_RUN_TOGGLE.value)
    sync_config_runtime_fields(CONFIG, APP_STATE)
    render_header()
    render_matrix_statuses()
    render_preview_output()


def on_run_clicked(_button) -> None:
    """Run the currently checked account/query groups."""
    snapshot = sync_runtime_selection(APP_STATE, collect_checked_groups(), DRY_RUN_TOGGLE.value)
    sync_config_runtime_fields(CONFIG, APP_STATE)

    with RUN_OUTPUT:
        clear_output(wait=True)
        if not snapshot["checked_groups"]:
            print("Select at least one account/query group.")
            return
        print(f"Running {len(snapshot['checked_groups'])} selected group(s).")
        print(f"Mode: {'DRY_RUN' if snapshot['dry_run'] else 'LIVE'}")
        print(f"Accounts: {', '.join(get_selected_accounts(snapshot['checked_groups']))}")

    APP_STATE["sanity_check_results"] = []
    APP_STATE["selected_events"] = []
    APP_STATE["payload_plan"] = {
        "selected_groups": [],
        "total_events": 0,
        "generated_at": None,
        "dry_run": snapshot["dry_run"],
    }
    APP_STATE["secops_result"] = None
    APP_STATE["query_results"] = build_initial_query_results(CONFIG, GROUP_CATALOG)
    sync_config_runtime_fields(CONFIG, APP_STATE)
    render_matrix_statuses()
    render_header()
    render_preview_output()

    sanity_results = run_selected_account_sanity_checks(
        CONFIG,
        get_selected_accounts(snapshot["checked_groups"]),
    )
    APP_STATE["sanity_check_results"] = sanity_results
    with RUN_OUTPUT:
        print("\nSanity checks:")
        display(pd.DataFrame(sanity_results))

    def progress_callback(group_key: str, result: dict) -> None:
        APP_STATE["query_results"][group_key] = result
        sync_config_runtime_fields(CONFIG, APP_STATE)
        render_matrix_statuses()

    results = execute_selected_groups(
        CONFIG,
        snapshot["checked_groups"],
        progress_fn=progress_callback,
    )
    for group_key, result in results.items():
        APP_STATE["query_results"][group_key] = result

    APP_STATE["checked_groups"] = snapshot["checked_groups"]
    APP_STATE["dry_run"] = snapshot["dry_run"]
    APP_STATE["last_run_selection"] = snapshot
    APP_STATE["selection_dirty"] = False
    APP_STATE["dirty_reason"] = "Ready to send."
    APP_STATE["selected_events"], APP_STATE["payload_plan"] = build_selected_events(
        APP_STATE["query_results"],
        APP_STATE["checked_groups"],
        dry_run=APP_STATE["dry_run"],
    )
    sync_config_runtime_fields(CONFIG, APP_STATE)

    result_rows = []
    for group_key in snapshot["checked_groups"]:
        result = APP_STATE["query_results"][group_key]
        result_rows.append(
            {
                "group_key": group_key,
                "status": result["status"],
                "rows": result["row_count"],
                "error": result["error"],
            }
        )

    with RUN_OUTPUT:
        print("\nExecution summary:")
        display(pd.DataFrame(result_rows))
        print(json.dumps(json_safe_value(summarize_query_results(APP_STATE["query_results"])), indent=2))

    render_header()
    render_matrix_statuses()
    render_preview_output()


def on_send_clicked(_button) -> None:
    """Send the currently checked successful groups to SecOps."""
    sync_runtime_selection(APP_STATE, collect_checked_groups(), DRY_RUN_TOGGLE.value)
    sync_config_runtime_fields(CONFIG, APP_STATE)
    send_state = reduce_send_state(APP_STATE)
    render_header()

    with SEND_OUTPUT:
        clear_output(wait=True)
        if not send_state["enabled"]:
            print(send_state["reason"])
            return

        events, payload_plan = build_selected_events(
            APP_STATE["query_results"],
            APP_STATE["checked_groups"],
            dry_run=APP_STATE["dry_run"],
        )
        APP_STATE["selected_events"] = events
        APP_STATE["payload_plan"] = payload_plan
        sender_config = build_sender_config(CONFIG, APP_STATE)
        sync_config_runtime_fields(CONFIG, APP_STATE)

        print(f"Sending {len(events)} event(s) from {len(payload_plan['selected_groups'])} group(s).")
        print(f"Mode: {'DRY_RUN' if sender_config['DRY_RUN'] else 'LIVE'}")
        print(f"Auth mode: {describe_auth_mode(sender_config['SECOPS_WEBHOOK_URL'])}")
        try:
            result = send_to_secops(events, sender_config)
        except Exception as exc:
            result = {
                "sent": 0,
                "batches": 0,
                "attempted_batches": 0,
                "failed_batches": 1,
                "failed_event_count": len(events),
                "planned_batches": 0,
                "planned_bytes": 0,
                "auth_mode": describe_auth_mode(sender_config["SECOPS_WEBHOOK_URL"]),
                "dry_run": sender_config["DRY_RUN"],
                "skipped": False,
                "errors": [{"error": str(exc)}],
            }
            print(f"Send failed: {exc}")

    APP_STATE["secops_result"] = result
    render_header()
    render_preview_output()


for checkbox in MATRIX_CHECKBOXES.values():
    checkbox.observe(refresh_selection_state, names="value")
DRY_RUN_TOGGLE.observe(refresh_selection_state, names="value")
RUN_BUTTON.on_click(on_run_clicked)
SEND_BUTTON.on_click(on_send_clicked)

render_header()
render_matrix_statuses()
render_preview_output()

display(
    widgets.VBox(
        [
            widgets.HTML("<h2>Snowflake Security Checks Control Panel</h2>"),
            HEADER_HTML,
            widgets.HBox([DRY_RUN_TOGGLE, RUN_BUTTON, SEND_BUTTON]),
            STATUS_HTML,
            MATRIX,
            OUTPUT_TABS,
        ]
    )
)


In [ ]:
# Cell 4 - Summary and debug
DEBUG_SNAPSHOT = build_debug_snapshot(CONFIG, APP_STATE)
print(json.dumps(DEBUG_SNAPSHOT, indent=2))
